[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/21_dimensionality_reduction/03_tsne_umap.ipynb)

# Bài 03 — t-SNE và UMAP: trực quan hoá phi tuyến

**🎯 Mục tiêu**
- Ý tưởng **manifold learning**: dữ liệu nhiều chiều thường nằm gần một "đa tạp" ít chiều, cong.
- **t-SNE**: giữ **láng giềng** — xác suất tương đồng ở không gian gốc (Gaussian) và không gian 2D (Student-t), tối thiểu KL divergence; vai trò của **perplexity**.
- **UMAP**: nhanh hơn, giữ cấu trúc toàn cục tốt hơn, có `transform` cho điểm mới.
- **Bẫy diễn giải**: kích thước cụm, khoảng cách giữa cụm, "cụm" từ nhiễu; đo bằng **trustworthiness**.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/21_dimensionality_reduction

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, trustworthiness

from ml_course.visualization import set_style

set_style()
rng = np.random.default_rng(0)
digits = load_digits()
warnings.filterwarnings("ignore", message="IProgress not found")
warnings.filterwarnings("ignore", message="n_jobs value")
X, y = digits.data, digits.target


def scatter_digits(ax, Z, title):
    for d in range(10):
        ax.scatter(*Z[y == d].T, s=5, color=plt.cm.tab10(d))
        ax.annotate(str(d), Z[y == d].mean(0), fontsize=14, weight="bold")
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])

## 1. PCA vs t-SNE trên digits

In [ ]:
start = time.perf_counter()
Z_tsne = TSNE(2, perplexity=30, random_state=0, init="pca").fit_transform(X)
t_tsne = time.perf_counter() - start
Z_pca = PCA(2).fit_transform(X)
fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
scatter_digits(axes[0], Z_pca, "PCA (tuyến tính)")
scatter_digits(axes[1], Z_tsne, f"t-SNE (perplexity 30, {t_tsne:.1f}s)")
plt.show()
for name, Z in [("PCA", Z_pca), ("t-SNE", Z_tsne)]:
    print(f"{name:6s} trustworthiness (k=10) = {trustworthiness(X, Z, n_neighbors=10):.3f}")

t-SNE tách các chữ số thành cụm rõ ràng. **Trustworthiness** ∈ [0, 1] đo mức độ các láng giềng trong 2D cũng là láng giềng thật trong không gian gốc (1 = hoàn hảo).

## 2. t-SNE hoạt động thế nào?

1. Không gian gốc: độ tương đồng $p_{j|i} \propto \exp(-\lVert\mathbf x_i - \mathbf x_j\rVert^2 / 2\sigma_i^2)$ — Gaussian quanh mỗi điểm, $\sigma_i$ chọn sao cho "số láng giềng hiệu dụng" = **perplexity**. Đối xứng hoá thành $p_{ij}$.
2. Không gian 2D: $q_{ij} \propto (1 + \lVert\mathbf z_i - \mathbf z_j\rVert^2)^{-1}$ — phân phối **Student-t** đuôi nặng (cho phép điểm xa nhau ở 2D mà không bị phạt nhiều → tránh "crowding problem").
3. Tìm $\mathbf z$ tối thiểu $\text{KL}(P \Vert Q) = \sum p_{ij}\log\frac{p_{ij}}{q_{ij}}$ bằng gradient descent.

KL **phạt nặng** khi láng giềng gần ($p$ lớn) bị đặt xa ($q$ nhỏ), nhưng **phạt nhẹ** khi điểm xa bị đặt gần → t-SNE giữ cấu trúc **cục bộ**, không giữ khoảng cách toàn cục.

### Perplexity

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(24, 5.5))
for ax, perp in zip(axes, [2, 10, 30, 100]):
    Zp = TSNE(2, perplexity=perp, random_state=0, init="pca").fit_transform(X)
    scatter_digits(ax, Zp, f"perplexity = {perp} | trust = {trustworthiness(X, Zp, n_neighbors=10):.3f}")
plt.show()

Perplexity nhỏ: chỉ nhìn vài láng giềng → nhiều cụm nhỏ vụn vặt. Perplexity lớn: nhìn rộng hơn → cấu trúc toàn cục mượt hơn. Thường thử 5–50; không có giá trị "đúng".

## 3. Bẫy diễn giải t-SNE

**(a) Kích thước và khoảng cách giữa các cụm vô nghĩa.** Hai cụm Gaussian: một cụm **rộng gấp 10 lần**, và một cụm nữa ở **rất xa**:

In [ ]:
A = rng.normal(0, 1, (200, 10))
B = rng.normal(0, 10, (200, 10)) + 40          # rộng gấp 10 lần
Cc = rng.normal(0, 1, (200, 10)) + 400         # rất xa
Xs = np.vstack([A, B, Cc])
ls = np.repeat([0, 1, 2], 200)
Zs = TSNE(2, perplexity=30, random_state=0).fit_transform(Xs)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for k, name in enumerate(["A (hẹp)", "B (rộng ×10)", "C (rất xa)"]):
    axes[0].scatter(*PCA(2).fit_transform(Xs)[ls == k].T, s=6, label=name)
    axes[1].scatter(*Zs[ls == k].T, s=6, label=name)
axes[0].set_title("PCA: giữ khoảng cách và độ rộng thật")
axes[1].set_title("t-SNE: cụm trông cùng cỡ, khoảng cách bị 'cân bằng'")
axes[0].legend()
plt.show()

**(b) "Cụm" từ nhiễu.** Dữ liệu ngẫu nhiên thuần (không có cụm nào) với perplexity nhỏ có thể trông như có cấu trúc:

In [ ]:
noise = rng.normal(size=(500, 50))
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, perp in zip(axes, [2, 30]):
    Zn = TSNE(2, perplexity=perp, random_state=0).fit_transform(noise)
    ax.scatter(*Zn.T, s=5)
    ax.set_title(f"Nhiễu thuần, perplexity = {perp}")
plt.show()

**(c) Ngẫu nhiên:** chạy lại với seed khác cho bố cục khác (cụm đổi chỗ, xoay). **(d) Không có `transform`:** t-SNE không ánh xạ được điểm mới; không dùng toạ độ t-SNE làm feature cho model.

> Quy tắc: dùng t-SNE/UMAP để **khám phá và trực quan hoá** (có những nhóm tách biệt không? có điểm lạ không?), không để đo khoảng cách, kích thước, hay làm feature — và kiểm chứng phát hiện bằng phương pháp khác.

## 4. UMAP

UMAP (McInnes et al., 2018) xây đồ thị láng giềng mờ (fuzzy) ở không gian gốc rồi tối ưu bố cục 2D giữ đồ thị đó (cross-entropy). Thực tế: **nhanh hơn** t-SNE nhiều, giữ cấu trúc **toàn cục** tốt hơn, có `transform` cho điểm mới. Tham số chính: `n_neighbors` (cục bộ ↔ toàn cục), `min_dist` (độ chặt của cụm).

In [ ]:
try:
    import umap

    fig, axes = plt.subplots(1, 3, figsize=(21, 6))
    for ax, nn in zip(axes, [5, 15, 100]):
        start = time.perf_counter()
        reducer = umap.UMAP(n_neighbors=nn, min_dist=0.1, random_state=0)
        Zu = reducer.fit_transform(X)
        scatter_digits(ax, Zu, f"UMAP n_neighbors={nn} ({time.perf_counter() - start:.1f}s) | trust = {trustworthiness(X, Zu, n_neighbors=10):.3f}")
    plt.show()
    Z_new = reducer.transform(X[:5])                                   # UMAP chiếu được điểm mới
    print("UMAP.transform cho 5 điểm mới:", Z_new.shape)
except ImportError:
    print("Chưa cài umap-learn — pip install umap-learn (phần UMAP được bỏ qua).")

## 5. Template: PCA vs t-SNE vs UMAP

| | PCA | t-SNE | UMAP |
|---|---|---|---|
| Loại | tuyến tính | phi tuyến | phi tuyến |
| Giữ | phương sai / khoảng cách toàn cục | láng giềng cục bộ | cục bộ + một phần toàn cục |
| Tốc độ | rất nhanh | chậm ($O(n\log n)$ Barnes-Hut) | nhanh |
| Điểm mới (`transform`) | có | không | có |
| Xác định (deterministic) | có | không (phụ thuộc seed) | không |
| Diễn giải trục | có (loadings) | không | không |
| Dùng làm feature | có | **không** | thận trọng |
| Dùng cho | tiền xử lý, nén, khử nhiễu, trực quan hoá nhanh | trực quan hoá | trực quan hoá, (tiền xử lý cho clustering) |

## 🧠 Kiểm tra nhanh

1. Vì sao t-SNE dùng phân phối Student-t trong không gian 2D?
2. Trên biểu đồ t-SNE, cụm A trông to gấp đôi cụm B. Có kết luận được cụm A phân tán hơn không?
3. Perplexity ảnh hưởng gì?
4. Vì sao không nên dùng toạ độ t-SNE làm feature cho model?

## 📌 Tóm tắt

- t-SNE/UMAP giữ láng giềng → trực quan hoá cấu trúc cụm phi tuyến tốt hơn PCA.
- Kích thước, khoảng cách giữa cụm, và hình dạng trên biểu đồ không đáng tin; kết quả phụ thuộc tham số và seed.
- UMAP nhanh hơn, có transform; PCA vẫn là lựa chọn cho tiền xử lý và nén.

## 📚 Tài liệu tham khảo

- van der Maaten & Hinton (2008) — *Visualizing Data using t-SNE*
- Wattenberg, Viégas & Johnson (2016) — *How to Use t-SNE Effectively* (distill.pub)
- McInnes, Healy & Melville (2018) — *UMAP: Uniform Manifold Approximation and Projection*